In [ ]:
"""
================================================================================
BigAlpha 2026 赛道一 (因子挖掘 / AI 智能子赛道) — CTDE-MARL 因子
================================================================================
架构: 中心化训练分散执行 (CTDE) 多智能体强化学习 (Multi-Agent RL)
  - Agent = 单只股票; 每个交易日 = 一步 (step)
  - 集中批评家 (Centralized Critic, 仅训练时): 看到"当日全市场截面分布",
    输出价值基线 V_t, 为策略梯度降方差 (CTDE 的 "Centralized Training")
  - 分散执行 (Decentralized Actor, 推理时): 每只股票仅用自身局部特征出因子,
    批评家不参与 (CTDE 的 "Decentralized Execution")
  - 奖励对齐 IC: 当日全市场 预测因子分位 vs 真实下期收益的 Spearman 相关 (Rank-IC)
  - LGBM 双重角色: (a) 筛子——用 gain 重要性保留 Top-K 因子去噪;
    (b) 信号灯—— slim LGBM 每日输出动态 Score_t 作为 RL 的核心状态特征
    (self-contained, 无文件桥; 训练信号用 OOF 防前视乐观)
  - 风险暴露表 exposure: 本版未纳入 RL 状态 (已压缩为 Score_t + 市场动态)

数据源 (写死, 与赛制一致): 官方 5 表
  bigalpha_2026_stock_bar1m / _financial / _instruments / _factorlib / _exposure

★ 本版关键修复:
  ① 架构: LGBM 作"筛子"(gain 重要性保留 Top-K) + "信号灯"(每日动态 Score_t);
     RL 状态 = Top-K 筛选因子(rank后) + Score_t (共 31 维, obs_dim=31, 修复此前 obs_dim=4 瓶颈)。
  ④ 训练稳定: 早停(按 MU_IC 最佳) + 50 轮 LR 衰减(1e-3→1e-4) + 60 轮 σ clamp 收紧(-4,-1),
     解决 RL_IC 在 ~90 轮见顶后退化振荡。
  ⑤ 保底因子: FACTOR_MODE='lgbm' 时直接输出无前视 OOF LGBM 信号 (等同 v9 基线, IC≈0.108),
     RL 坍缩时自动回退, 确保 ipynb 稳定吐出合规因子。
  ⑥ 组合奖励修正 (corr≈1 分支): 奖励改为 IC(lgbm_z+λ·act_z, ret), 关掉 supervised_coef,
     早停口径改为验证集组合 IC, 训练标签做风格中性化 — 逼 RL 学 LGBM 之上的真残差。
  ② 梯度为零 bug: 原奖励算在确定性 mu 上 → A 与动作 a 独立 → ∇μ pol_loss=0,
     μ 永远学不到。修复: 奖励算在采样动作 act 上 (A 依赖 a) → ∇μ logp=(act-μ)/σ²≠0。
  ③ 训练信号前视乐观: 训练集 lgbm_sig 改用 OOF(4折)生成, 避免 RL 在"先知特征"上训练;
     测试集用固定模型 predict, 诚实样本外。

子赛道归属: AI 智能 (强化学习) — 满足"AI 参与度"合规, 非独立树模型提交
提交格式: main(datasources, start_date, end_date) → date,instrument,factor
================================================================================
"""

try:
    from bigmodule import M, I
except ImportError:
    M = I = None

import os
import copy
import math
import warnings
import numpy as np
import pandas as pd

try:
    import dai
except ImportError:
    dai = None

import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── CPU 环境约束: 16 cores / 32GB ──
# LGBM 与 torch 训练顺序执行, 不会同时抢核. torch 小网络(隐层64)线程开销敏感,
# 给 8 线程留余量; LGBM 用 12 线程吃满剩余算力. 均不拉满 16 以防平台自身开销争抢.
torch.set_num_threads(8)
CTDE_EPOCHS = 100   # 上限; RL_IC 在 80-110 轮见顶, 配合早停在 ~90-100 轮最佳处停止
TOPK = 30            # LGBM gain 重要性筛选后保留的 Top-K 因子, 再训 slim LGBM 出 Score_t

# ── 训练策略 (针对 RL_IC 在 ~90 轮见顶、120 轮后退化振荡) ──
LR = 1e-3
LR_DECAY_EPOCH = 50     # 50 轮后 lr 降至 1e-4, 防后期振荡跳出局部优
LR_DECAY = 1e-4
SIG_TIGHTEN_EPOCH = 60  # 60 轮后 logsig clamp 收紧 (-4,-1), 强制确定性增强
EARLY_STOP_PATIENCE = 25  # 连续 25 轮 MU_IC(部署指标) 无改善则早停
FACTOR_MODE = "composite"  # "composite"=RL+LGBM 按IC加权融合(默认); "rl"=仅RL; "lgbm"=仅OOF LGBM(保底)
COMBINE_LAMBDA = 1.0   # 组合奖励目标: IC(lgbm_z + λ·act_z, ret); λ 控制 RL 在 LGBM 之上的增量权重

# ── 数据源表名 (写死, 与赛制一致) ──
FIN_TABLE = "bigalpha_2026_financial"
BAR1M_TABLE = "bigalpha_2026_stock_bar1m"
LIB_TABLE = "bigalpha_2026_factorlib"
INST_TABLE = "bigalpha_2026_instruments"
EXP_TABLE = "bigalpha_2026_exposure"

# ── 特征常量 ──
PRICE_COLS = ["open", "high", "low", "close", "volume", "amount"]
FIN_COLS = [
    "operating_revenue", "net_profit_to_parent_shareholders",
    "total_assets", "total_equity_to_parent_shareholders",
]
EXCLUDE_LIB = {
    "date", "instrument", "close", "open", "high", "low",
    "volume", "amount", "report_date", "shift", "category",
    "name", "adjust_factor", "pre_close", "instrument_id",
}
EXP_EXCLUDE = {"date", "instrument", "instrument_id", "report_date", "name"}
LIB_WHITELIST = {
    "rsi_12", "cci_14", "bias_20",
    "macd_hist_12_26_9", "macd_dea_12_26_9",
    "volatility_5", "roe_avg_ttm", "roa_avg_ttm",
    "gross_profit_rate_ttm", "net_profit_rate_ttm",
    "debt_to_asset_lf", "float_market_cap", "pe_ttm", "ps_ttm", "pb",
    "netflow_amount_main", "netflow_amount_rate_main", "net_active_buy_amount_main",
    "amount", "volume",
}
DERIVED = ["ret_1", "ret_5", "ret_20", "vol_5", "vol_20",
           "range_1", "intraday_rev", "volume_z", "amount_z"]

# ── 风格/行业中性化基准 (用于训练标签中性化, 对齐官方"风格剔除") ──
STYLE_HINTS = {
    "r_total_assets", "r_total_equity_to_parent_shareholders", "r_float_market_cap",
    "r_BETA", "r_MEDIA", "r_LIQUIDTY", "r_HEALTH", "r_UTILITIES", "r_TELECOM",
    "r_FOODBEVER", "r_ELECEQP", "r_REALESTATE", "r_ELECTRONICS", "r_AUTO",
    "r_CHEM", "r_COMPUTER", "r_NONFERMETAL", "r_TRANSPORTATION", "r_RESVOL",
}


# ════════════════════════════════════════════════════════════════
# 1. 特征工程 (训练/预测共用) — v9 同款数据层
# ════════════════════════════════════════════════════════════════

def _cs_winsorize(df, cols, lo=0.01, hi=0.99):
    g = df.groupby("date")
    for c in cols:
        if c in df.columns:
            df[c] = df[c].replace([np.inf, -np.inf], np.nan)
            df[c] = df[c].clip(g[c].transform("quantile", q=lo),
                               g[c].transform("quantile", q=hi))
    return df


def _cs_zscore(df, cols):
    g = df.groupby("date")
    for c in cols:
        if c in df.columns:
            df[c] = (df[c] - g[c].transform("mean")) / g[c].transform("std").replace(0, 1.0)
    return df


def build_features(financial_table, bar1m_table, sd, ed, price_start=None):
    """加载 [sd, ed] 行情/财务/LOB/factorlib/exposure, 返回 (df, raw_feats)。
    复用 v9 的衍生特征 + LOB + factorlib 白名单逻辑, 新增 exposure 风险暴露。"""
    sd_ts, ed_ts = pd.Timestamp(sd), pd.Timestamp(ed)
    fin_start = sd_ts - pd.Timedelta(days=365)

    # ── 财务 PIT ──
    fin = dai.query(
        f"SELECT date,instrument,{','.join(FIN_COLS)} "
        f"FROM {financial_table} WHERE category='lf' AND shift=0",
        filters={"date": [fin_start, ed_ts]}, compression=True,
    ).df()
    fin["date"] = pd.to_datetime(fin["date"])
    fin["instrument"] = fin["instrument"].astype(str)
    nd = pd.date_range(fin_start, ed_ts)
    fin = (
        fin.set_index("date").groupby("instrument", group_keys=False)
        .apply(lambda g: g.reindex(nd).ffill().assign(instrument=g.name))
        .reset_index().rename(columns={"index": "date"}).dropna(subset=["instrument"])
    )
    fin["date"] = pd.to_datetime(fin["date"])
    for c in FIN_COLS:
        fin[c] = pd.to_numeric(fin[c], errors="coerce")

    # ── 量价 (分钟→日频) ──
    price_buf = pd.Timestamp(price_start) if price_start is not None else sd_ts - pd.Timedelta(days=365)
    price = dai.query(
        f"""SELECT date::DATE::DATETIME AS date,instrument::string AS instrument,
            ARG_MIN(open,date) AS open,MAX(high) AS high,MIN(low) AS low,
            ARG_MAX(close,date) AS close,SUM(volume) AS volume,SUM(amount) AS amount
            FROM {bar1m_table} GROUP BY 1,2 ORDER BY 1,2""",
        filters={"date": [price_buf, ed_ts]}, compression=True,
    ).df()
    price["date"] = pd.to_datetime(price["date"])
    price["instrument"] = price["instrument"].astype(str)
    for c in PRICE_COLS:
        price[c] = pd.to_numeric(price[c], errors="coerce")
    price = price.sort_values(["instrument", "date"]).reset_index(drop=True)

    g = price.groupby("instrument", group_keys=False)
    price["ret_1"] = g["close"].pct_change(1)
    price["ret_5"] = g["close"].pct_change(5)
    price["ret_20"] = g["close"].pct_change(20)
    r = g["close"].pct_change()
    price["vol_5"] = r.transform(lambda x: x.rolling(5, min_periods=3).std())
    price["vol_20"] = r.transform(lambda x: x.rolling(20, min_periods=10).std())
    price["range_1"] = (price["high"] - price["low"]) / price["close"].replace(0, np.nan)
    price["intraday_rev"] = (price["close"] - price["open"]) / price["open"].replace(0, np.nan)
    price["volume_z"] = (
        np.log1p(price["volume"]) - g["volume"].transform(lambda x: np.log1p(x).rolling(20, min_periods=5).mean())
    ) / g["volume"].transform(lambda x: np.log1p(x).rolling(20, min_periods=5).std()).replace(0, np.nan)
    price["amount_z"] = (
        np.log1p(price["amount"]) - g["amount"].transform(lambda x: np.log1p(x).rolling(20, min_periods=5).mean())
    ) / g["amount"].transform(lambda x: np.log1p(x).rolling(20, min_periods=5).std()).replace(0, np.nan)
    price["label"] = g["close"].shift(-1) / price["close"] - 1      # T+1 收益 (合规 label)

    df = pd.merge(price, fin, how="inner", on=["date", "instrument"])

    # ── LOB 十档 (官方 spec: bid/ask 各 10 档) ──
    LOB_COLS = ["lob_imb", "lob_depth"]
    try:
        lob = dai.query(
            f"""SELECT date::DATE::DATETIME AS date, instrument::string AS instrument,
                AVG((
                    bid_volume1*1.0  + bid_volume2*0.741 + bid_volume3*0.549 + bid_volume4*0.407 + bid_volume5*0.301
                    + bid_volume6*0.223 + bid_volume7*0.165 + bid_volume8*0.122 + bid_volume9*0.091 + bid_volume10*0.067
                    - ask_volume1*1.0  - ask_volume2*0.741 - ask_volume3*0.549 - ask_volume4*0.407 - ask_volume5*0.301
                    - ask_volume6*0.223 - ask_volume7*0.165 - ask_volume8*0.122 - ask_volume9*0.091 - ask_volume10*0.067
                ) / NULLIF(
                    bid_volume1+bid_volume2+bid_volume3+bid_volume4+bid_volume5+bid_volume6+bid_volume7+bid_volume8+bid_volume9+bid_volume10
                    + ask_volume1+ask_volume2+ask_volume3+ask_volume4+ask_volume5+ask_volume6+ask_volume7+ask_volume8+ask_volume9+ask_volume10
                    , 0)) AS lob_imb,
                AVG((
                    bid_volume1+bid_volume2+bid_volume3+bid_volume4+bid_volume5+bid_volume6+bid_volume7+bid_volume8+bid_volume9+bid_volume10
                    - ask_volume1-ask_volume2-ask_volume3-ask_volume4-ask_volume5-ask_volume6-ask_volume7-ask_volume8-ask_volume9-ask_volume10
                ) * 1.0 / NULLIF(
                    bid_volume1+bid_volume2+bid_volume3+bid_volume4+bid_volume5+bid_volume6+bid_volume7+bid_volume8+bid_volume9+bid_volume10
                    + ask_volume1+ask_volume2+ask_volume3+ask_volume4+ask_volume5+ask_volume6+ask_volume7+ask_volume8+ask_volume9+ask_volume10
                    , 0)) AS lob_depth
                FROM {bar1m_table} WHERE ask_price1>0 AND bid_price1>0 GROUP BY 1,2""",
            filters={"date": [price_buf, ed_ts]}, compression=True,
        ).df()
        lob["date"] = pd.to_datetime(lob["date"])
        lob["instrument"] = lob["instrument"].astype(str)
        for c in LOB_COLS:
            lob[c] = pd.to_numeric(lob[c], errors="coerce")
        df = pd.merge(df, lob, how="left", on=["date", "instrument"])
    except Exception:
        pass

    # ── factorlib 白名单 ──
    lib_cols = []
    try:
        flib = dai.query(f"SELECT * FROM {LIB_TABLE}", filters={"date": [price_buf, ed_ts]}, compression=True).df()
        flib["date"] = pd.to_datetime(flib["date"])
        flib["instrument"] = flib["instrument"].astype(str)
        cols_all = [c for c in flib.columns if c not in EXCLUDE_LIB]
        if LIB_WHITELIST:
            cols_all = [c for c in cols_all if c in LIB_WHITELIST]
        lib_cols = [c for c in cols_all if c not in df.columns]
        flib = flib[["date", "instrument"] + lib_cols]
        for c in lib_cols:
            flib[c] = pd.to_numeric(flib[c], errors="coerce")
        df = pd.merge(df, flib, how="left", on=["date", "instrument"])
    except Exception:
        lib_cols = []

    # ── 风险暴露 exposure (新增) ──
    exp_cols = []
    try:
        exp = dai.query(f"SELECT * FROM {EXP_TABLE}", filters={"date": [price_buf, ed_ts]}, compression=True).df()
        exp["date"] = pd.to_datetime(exp["date"])
        exp["instrument"] = exp["instrument"].astype(str)
        exp_cols_all = [c for c in exp.columns if c not in EXP_EXCLUDE]
        exp_cols = [c for c in exp_cols_all if c not in df.columns]
        if exp_cols:
            exp = exp[["date", "instrument"] + exp_cols]
            for c in exp_cols:
                exp[c] = pd.to_numeric(exp[c], errors="coerce")
            df = pd.merge(df, exp, how="left", on=["date", "instrument"])
    except Exception:
        exp_cols = []

    raw_feats = list(DERIVED) + list(FIN_COLS)
    if any(c in df.columns for c in LOB_COLS):
        raw_feats += [c for c in LOB_COLS if c in df.columns]
    if lib_cols:
        raw_feats += [c for c in lib_cols if c not in raw_feats]
    if exp_cols:
        raw_feats += [c for c in exp_cols if c not in raw_feats]
    raw_feats = list(dict.fromkeys([c for c in raw_feats if c in df.columns]))

    for c in raw_feats:
        df[c] = df[c].replace([np.inf, -np.inf], np.nan)
        df[c] = df.groupby("date")[c].transform(lambda x: x.fillna(x.median()))
        df[c] = df[c].fillna(0)

    df = df[(df["date"] >= sd_ts) & (df["date"] <= ed_ts)].reset_index(drop=True)
    return df, raw_feats


# ════════════════════════════════════════════════════════════════
# 2. LGBM-v9 信号 (内存内训练, 作为 actor 的一路输入特征)
# ════════════════════════════════════════════════════════════════

def train_lgbm_signal(df, rank_feats, label="label"):
    """内存中 LightGBM(huber, 即 v9 基线) 回归下期收益, 返回训练好的 booster。
    仅作特征源, 不单独提交。模型在训练集拟合一次, 预测跨窗口复用 (无泄漏)。"""
    import lightgbm as lgb
    sub = df.dropna(subset=rank_feats + [label]).copy()
    model = lgb.LGBMRegressor(
        objective="huber", n_estimators=200, learning_rate=0.03,
        max_depth=5, subsample=0.8, colsample_bytree=0.7,
        min_child_samples=40, reg_lambda=3.0, random_state=SEED, n_jobs=12, verbose=-1,
    )
    model.fit(sub[rank_feats], sub[label])
    return model


def add_lgbm_signal(df, model, rank_feats):
    """用已训练 LGBM 预测截面信号 'lgbm_sig', 按日标准化 (仅推理, 不接触标签)。"""
    for f in rank_feats:
        if f not in df.columns:
            df[f] = 0.0
    df[rank_feats] = df[rank_feats].fillna(0.0)
    df["lgbm_sig"] = model.predict(df[rank_feats])

    def _z(x):
        s = x.std()
        if s == 0 or np.isnan(s):   # 标量, 不能用 .replace
            s = 1.0
        return (x - x.mean()) / s

    df["lgbm_sig"] = df.groupby("date")["lgbm_sig"].transform(_z)
    return df


def select_topk_features(model, feats, k):
    """用 LGBM gain 重要性排序, 返回 Top-K 特征名 (静态筛子, 只看一次)。"""
    booster = model.booster_
    imp = pd.Series(booster.feature_importance(importance_type="gain"), index=feats)
    imp = imp.sort_values(ascending=False)
    return list(imp.head(k).index)


def add_lgbm_signal_oof(df, feats, label="label", n_splits=4):
    """训练集用 n 折 OOF 生成 lgbm_sig, 避免 RL 在 in-sample 先知信号上训练
    (每折用其余折训练、对该折预测, 杜绝该折标签泄露到自身信号)。"""
    sub = df.dropna(subset=feats + [label]).copy()
    oof = np.full(len(sub), np.nan)
    idx = np.arange(len(sub))
    rng = np.random.RandomState(SEED)
    rng.shuffle(idx)
    folds = np.array_split(idx, n_splits)
    for fi, val_idx in enumerate(folds):
        tr_idx = np.setdiff1d(idx, val_idx)
        print(f"    OOF fold {fi+1}/{n_splits} (train={len(tr_idx)}, val={len(val_idx)})...")
        m = train_lgbm_signal(sub.iloc[tr_idx], feats, label)
        oof[val_idx] = m.predict(sub.iloc[val_idx][feats])
    out = df.copy()
    out["lgbm_sig"] = pd.Series(oof, index=sub.index).reindex(df.index)
    out["lgbm_sig"] = out.groupby("date")["lgbm_sig"].transform(
        lambda x: (x - x.mean()) / (x.std() if x.std() and not np.isnan(x.std()) else 1.0)
    )
    return out


# ════════════════════════════════════════════════════════════════
# 3. CTDE 网络: 分散 Actor + 集中 Critic
# ════════════════════════════════════════════════════════════════

class DecentralizedActor(nn.Module):
    """每只股票独立策略 (推理时只看局部特征)。
    输出: μ(factor均值), logσ(探索)。"""

    def __init__(self, obs_dim, hidden=32):
        super().__init__()
        # ★ 缩小容量 + Dropout 防过拟合 (31维输入 → 32 → 16; 训练集 IC 虚高, 控容量提 OOS)
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.2),
        )
        self.mu_head = nn.Linear(hidden // 2, 1)
        self.logsig_head = nn.Linear(hidden // 2, 1)
        self.sig_clamp = (-3, 1)   # 训练后期由主循环收紧到 (-4,-1)
        self.hidden = hidden

    def forward(self, x):
        h = self.net(x)
        mu = self.mu_head(h)
        log_sig = self.logsig_head(h).clamp(*self.sig_clamp)
        return mu, log_sig


class CentralizedCritic(nn.Module):
    """集中批评家 (仅训练时): 输入当日全市场截面分布向量 (7×obs_dim),
    输出价值基线 V_t。CTDE 的 'Centralized' —— 看到全局状态。"""

    def __init__(self, global_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(global_dim, hidden), nn.ReLU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, g):
        return self.net(g)


def _global_vector(obs_mat):
    """把当日 (N, D) 截面特征压缩为固定维度全局向量 (分位数摘要)。"""
    if obs_mat.shape[0] == 0:
        return np.zeros(obs_mat.shape[1] * 7, dtype=np.float32)
    q = np.quantile(obs_mat, [0.05, 0.25, 0.5, 0.75, 0.95], axis=0)
    mean = obs_mat.mean(axis=0)
    std = obs_mat.std(axis=0)
    return np.concatenate([mean, std, q[0], q[1], q[2], q[3], q[4]]).astype(np.float32)


# ════════════════════════════════════════════════════════════════
# 4. CTDE 训练 (REINFORCE/A2C + 集中基线)
#    ★ 组合奖励 (决策树 corr≈1 分支): 目标 IC(lgbm_z + λ·act_z, ret)
#      —— RL 只有在 LGBM 之上额外增加信息时才得分, 天然学真残差, 不复制 LGBM
#    ★ 梯度: 采样动作 act 断图 + logp 引用 (act,μ) → ∇μ logp=(act-μ)/σ²≠0
#      (切勿改成重参数化 act=μ+σ·eps 不 detach —— 那样 logp 里 μ 消失, μ 梯度归零)
#    ★ 早停/最佳模型按验证集【组合】IC(IC(lgbm_z+λ·μ_z, ret)) 决定, 对齐最终提交目标
#    ★ supervised_coef 默认 0: 关闭会强迫 μ→lgbm 的辅助项 (那是上一版 corr→1 的根因)
# ════════════════════════════════════════════════════════════════

def ctde_train(actor, critic, train_episodes, val_episodes=None, epochs=100, lr=1e-3,
               device="cpu", value_coef=0.5, lr_decay_epoch=50, lr_decay=1e-4,
               sig_tighten_epoch=60, patience=25, eval_every=5, supervised_coef=0.0):
    """CTDE 训练 + 验证集早停 + LR 衰减 + σ 收紧。
    ★ 组合奖励: 主目标 IC(lgbm_z + λ·act_z, ret) —— RL 仅在 LGBM 之上额外增加信息时才得分,
      天然学真残差 (不再复制 LGBM); supervised_coef 默认 0 (关闭会强迫 μ→lgbm 的辅助项)。
    早停/最佳模型按验证集【组合】IC(IC(lgbm_z+λ·μ_z, ret)) 决定, 对齐最终提交目标。
    返回 (actor, critic, best_val_ic)。"""
    actor, critic = actor.to(device), critic.to(device)
    opt_a = torch.optim.AdamW(actor.parameters(), lr=lr)
    opt_c = torch.optim.AdamW(critic.parameters(), lr=lr)
    mse = nn.MSELoss()

    def _mu_ic(eps):
        """确定性 μ 与收益的 IC —— 即最终提交因子的 IC (部署/验证监控统一口径)。"""
        actor.eval()
        preds, rets = [], []
        with torch.no_grad():
            for e in eps:
                mu, _ = actor(e["obs"].to(device))
                preds.append(mu.cpu().numpy().flatten())
                rets.append(e["ret"].numpy().flatten())
        if not preds:
            return 0.0
        ic = spearmanr(np.concatenate(preds), np.concatenate(rets))[0]
        return 0.0 if np.isnan(ic) else ic

    def _combined_ic(eps):
        """组合因子 IC: IC(lgbm_z + λ·μ_z, ret) —— 最终提交复合因子的验证 IC (早停口径)。"""
        actor.eval()
        preds, rets = [], []
        with torch.no_grad():
            for e in eps:
                mu, _ = actor(e["obs"].to(device))
                mu_s = mu.squeeze(-1).cpu().numpy().flatten()
                mu_z = (mu_s - mu_s.mean()) / (mu_s.std() + 1e-6)
                lgbm = e["lgbm"].numpy().astype(np.float32).flatten()
                lgbm_z = (lgbm - lgbm.mean()) / (lgbm.std() + 1e-6)
                preds.append(lgbm_z + COMBINE_LAMBDA * mu_z)
                rets.append(e["ret"].numpy().flatten())
        if not preds:
            return 0.0
        ic = spearmanr(np.concatenate(preds), np.concatenate(rets))[0]
        return 0.0 if np.isnan(ic) else ic

    best_val_ic = -1.0
    best_state = None
    no_improve = 0

    for ep in range(epochs):
        # ── 调度: LR 衰减 / σ 收紧 ──
        if ep == lr_decay_epoch:
            for pg in opt_a.param_groups:
                pg["lr"] = lr_decay
            for pg in opt_c.param_groups:
                pg["lr"] = lr_decay
            print(f"  ↓ LR 衰减至 {lr_decay} @ epoch {ep+1}")
        if ep == sig_tighten_epoch:
            actor.sig_clamp = (-4, -1)
            print(f"  ↓ σ clamp 收紧至 (-4,-1) @ epoch {ep+1}")

        # ── 训练一个 epoch (仅用 train_episodes, 不泄漏验证集) ──
        actor.train(); critic.train()
        tot_p, tot_v, tot_e, tot_rl, tot_sup = 0.0, 0.0, 0.0, 0.0, 0.0
        n_dates = 0
        opt_a.zero_grad(); opt_c.zero_grad()
        for e in train_episodes:
            obs = e["obs"].to(device)              # (N, D)
            ret = e["ret"].numpy().astype(np.float32)
            gv = e["global"].to(device)

            mu, log_sig = actor(obs)
            sig = torch.exp(log_sig).clamp(min=1e-3)
            # ★ REINFORCE 正确梯度: act 断图, 奖励算在 act 上, logp 引用 (act,μ)
            act = (mu + sig * torch.randn_like(mu)).detach()

            # ★ 组合奖励: RL 仅在 LGBM 之上额外增加信息时才得分 → 学真残差, 不复制 LGBM
            #   ★ 维度修复: lgbm_t/act 必须都压成 1 维, 否则 (N,) + (N,1) 广播成 (N,N) 导致
            #     spearmanr 维度不匹配报错. act 本体保留 (N,1) 给下方 logp 梯度路径.
            lgbm_t = e["lgbm"].to(device).flatten()
            lgbm_z = (lgbm_t - lgbm_t.mean()) / (lgbm_t.std() + 1e-6)
            act_f = act.flatten()
            act_z = (act_f - act_f.mean()) / (act_f.std() + 1e-6)
            combined = lgbm_z + COMBINE_LAMBDA * act_z
            ic = spearmanr(combined.cpu().numpy().flatten(), ret)[0]
            if np.isnan(ic):
                ic = 0.0
            R = torch.tensor([ic], dtype=torch.float32, device=device)

            V = critic(gv).squeeze(-1)             # (1,)
            A = (R - V).detach()                   # 集中基线降方差

            logp = -0.5 * ((act - mu) / sig) ** 2 - log_sig - 0.5 * math.log(2 * math.pi)
            ent = (log_sig + 0.5 * math.log(2 * math.pi) + 0.5).mean()
            pol_loss = -(A * logp).mean()
            val_loss = mse(V, R)

            # ★ supervised_coef 默认 0: 关闭会强迫 μ→lgbm 的辅助项 (那是 corr→1 复制的根因);
            #   组合奖励已让 RL 在 LGBM 之上做增量, 不再需要行为克隆先验
            sup_loss = torch.zeros((), device=device)
            if supervised_coef > 0:
                mu_s = mu.squeeze(-1)
                mu_z = (mu_s - mu_s.mean()) / (mu_s.std() + 1e-6)
                lgbm_target = e["lgbm"].to(device)
                sup_loss = F.mse_loss(mu_z, lgbm_target)

            loss = pol_loss + value_coef * val_loss + supervised_coef * sup_loss

            loss.backward()
            tot_p += pol_loss.item(); tot_v += val_loss.item()
            tot_e += ent.item(); tot_rl += ic; tot_sup += sup_loss.item()
            n_dates += 1

        torch.nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
        torch.nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
        opt_a.step(); opt_c.step()

        # 训练集监控 (仅供参考, 不用于早停)
        rl_ic_avg = tot_rl / n_dates
        mu_ic_tr = _mu_ic(train_episodes)
        if (ep + 1) % eval_every == 0 or ep == 0:
            print(f"  Epoch {ep+1:3d}/{epochs}: pol={tot_p/n_dates:.4f} "
                  f"val={tot_v/n_dates:.4f} ent={tot_e/n_dates:.4f} "
                  f"sup={tot_sup/n_dates:.4f} "
                  f"RL_IC(train)={rl_ic_avg:.4f} MU_IC(train)={mu_ic_tr:.4f} "
                  f"COMB_IC(train)={_combined_ic(train_episodes):.4f}")

        # ── 验证集评估 + 早停 (每 eval_every 轮, 口径=组合 IC) ──
        if val_episodes and ((ep + 1) % eval_every == 0 or ep == 0):
            val_cic = _combined_ic(val_episodes)
            val_mic = _mu_ic(val_episodes)
            print(f"    [val] COMB_IC={val_cic:.4f} MU_IC={val_mic:.4f} best={best_val_ic:.4f}")
            if val_cic > best_val_ic + 1e-6:
                best_val_ic = val_cic
                best_state = (copy.deepcopy(actor.state_dict()),
                              copy.deepcopy(critic.state_dict()))
                no_improve = 0
            else:
                no_improve += 1
                if no_improve * eval_every >= patience:
                    print(f"  🛑 早停 @ epoch {ep+1} (验证 COMB_IC 连续 {no_improve*eval_every} 轮无改善, 最佳={best_val_ic:.4f})")
                    break

    if best_state is None:
        # 验证集为空或全程未刷新: 退回最后一个 epoch
        best_state = (copy.deepcopy(actor.state_dict()), copy.deepcopy(critic.state_dict()))
        best_val_ic = _combined_ic(val_episodes) if val_episodes else 0.0
    actor.load_state_dict(best_state[0])
    critic.load_state_dict(best_state[1])
    print(f"  ✅ 恢复最佳模型 (验证 COMB_IC={best_val_ic:.4f})")
    return actor.cpu(), critic.cpu(), best_val_ic


# ════════════════════════════════════════════════════════════════
# 5. 入口 main() —— 赛方一键运行
# ════════════════════════════════════════════════════════════════

# ── 平台 datasources 容错解析 (兼容 dict 多种 key / DAI Dataset 对象) ──
def _as_table_name(x):
    """把 datasources 的值规整为表名字符串。"""
    if isinstance(x, str):
        return x
    for attr in ("table_name", "name", "_name"):
        if hasattr(x, attr):
            v = getattr(x, attr)
            if isinstance(v, str) and v:
                return v
    s = str(x)
    for t in (FIN_TABLE, BAR1M_TABLE, LIB_TABLE, INST_TABLE, EXP_TABLE):
        if t in s:
            return t
    return s


def _resolve_table(datasources, *candidates):
    """按多种可能的 key 在 datasources 中解析表名; 找不到返回 None。"""
    if isinstance(datasources, dict):
        for c in candidates:
            if c in datasources and datasources[c] is not None:
                return _as_table_name(datasources[c])
    return None


def _build_factor_output(te_df, factor_col, stk_pool, tag=""):
    """v9 范式: 截面标准化→对齐成分股→返回合规 [date,instrument,factor] DataFrame。

    无论 RL 通道还是 LGBM 通道, 都经此函数产出各自独立、对齐成分股后的合规因子,
    确保两通道都 '吐出合规 DataFrame、对齐成分股'(赛制单因子提交时再择一 return)。
    """
    out = te_df[["date", "instrument", factor_col]].rename(columns={factor_col: "factor"})
    # 截面标准化到 [-1, 1] (与 v9 同款: rank(pct)*2-1)
    out["factor"] = out.groupby("date")["factor"].rank(pct=True) * 2 - 1
    # 对齐成分股 (inner join bigalpha_2026_instruments)
    out = pd.merge(out, stk_pool, how="inner", on=["date", "instrument"])
    out["factor"] = out["factor"].replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["factor"]).reset_index(drop=True)
    out["date"] = pd.to_datetime(out["date"])
    out["instrument"] = out["instrument"].astype(str)
    # 诊断
    f = out["factor"]
    print(f"  [{tag}] 合规因子: {len(out)} 行, {out['instrument'].nunique()} 只, "
          f"std={f.std():.4f} 唯一值={f.nunique()} 接近0占比={(f.abs() < 0.01).mean():.3f}")
    return out[["date", "instrument", "factor"]]


def _plot_weights(full_model, rank_feats, topk, w_rl, w_lgbm, mu_ic, lgbm_ic):
    """生成权重图并保存:
    左: 被选中的 Top-K 特征的 LGBM gain 重要性 (展示'为什么选这30个'及相对权重);
    右: 复合因子融合权重 (RL vs LGBM, 按训练集 IC 加权)。
    平台 notebook 内联显示; 本地无 matplotlib 时跳过。"""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        import os as _os
        booster = full_model.booster_
        imp = pd.Series(booster.feature_importance(importance_type="gain"), index=rank_feats)
        imp = imp.sort_values(ascending=False)
        topk_imp = imp.reindex([c for c in topk if c in imp.index])
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
        ax1.barh(range(len(topk_imp)), topk_imp.values, color="#2563eb")
        ax1.set_yticks(range(len(topk_imp)))
        ax1.set_yticklabels(topk_imp.index, fontsize=7)
        ax1.invert_yaxis()
        ax1.set_title(f"Top-{len(topk_imp)} 特征 LGBM gain 重要性 (筛选模型·子采样20%)")
        ax1.set_xlabel("gain")
        ax2.bar(["RL\n(MU_IC=%.4f)" % mu_ic, "LGBM\n(IC=%.4f)" % lgbm_ic],
                [w_rl, w_lgbm], color=["#2563eb", "#16a34a"])
        ax2.set_ylim(0, 1)
        ax2.set_title("复合因子融合权重")
        ax2.set_ylabel("weight")
        for i, v in enumerate([w_rl, w_lgbm]):
            ax2.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11)
        fig.tight_layout()
        out_p = "composite_weights.png"
        fig.savefig(out_p, dpi=120, bbox_inches="tight")
        plt.close(fig)
        print(f"  💡 权重图已保存: {_os.path.abspath(out_p)}")
    except Exception as e:
        print(f"  ⚠️ 权重图生成跳过 (matplotlib 不可用或异常): {e}")


def main(datasources, start_date, end_date):
    print("=" * 70)
    print("BigAlpha 2026 赛道一 (AI) — CTDE-MARL 因子 + LGBM-v9 信号 + exposure")
    print("=" * 70)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    TRAIN_START, TRAIN_END = "2022-01-01 00:00:00", "2023-12-31 23:59:59"

    # ── 1. 训练集 ──
    print("\n[1/5] 构建训练集 (2022-2023)...")
    tr_df, raw_feats = build_features(FIN_TABLE, BAR1M_TABLE, TRAIN_START, TRAIN_END,
                                      price_start=TRAIN_START)
    tr_df = _cs_winsorize(tr_df, raw_feats + ["label"])
    tr_df = _cs_zscore(tr_df, raw_feats + ["label"])
    for c in raw_feats:
        tr_df["r_" + c] = tr_df.groupby("date")[c].rank(pct=True)
    rank_feats = ["r_" + c for c in raw_feats if "r_" + c in tr_df.columns]
    tr_df = tr_df.dropna(subset=rank_feats + ["label"]).reset_index(drop=True)

    # ── 2. LGBM 双重角色: 筛子(重要性) + 信号灯(Score_t) ──
    print("[2/5] LGBM 筛选 Top-K 因子 + 训练 slim 信号灯...")
    print("  [2.1] 重要性筛选模型 (子采样20%, 仅看重要性, 提速)...")
    sub_imp = tr_df.sample(frac=0.2, random_state=SEED)
    full_model = train_lgbm_signal(sub_imp, rank_feats)      # 仅用于看重要性 (子采样)
    topk = select_topk_features(full_model, rank_feats, TOPK) # 筛子: 保留 Top-K
    print(f"  Top-{TOPK} 因子: {topk}")
    print("  [2.2] 训练 slim 信号灯 (全训练集)...")
    slim_model = train_lgbm_signal(tr_df, topk)              # 部署用固定模型 (全训练集)
    print("  [2.3] 训练信号 OOF (4折, 防前视)...")
    tr_df = add_lgbm_signal_oof(tr_df, topk, n_splits=4)     # 训练信号用 OOF, 诚实不泄
    # RL 状态 = 筛选出的 Top-K 因子(rank 后) + LGBM 动态 Score_t
    # ★ 修正: 此前误只取 3 个原始 rank 特征 → obs_dim=4 信息瓶颈, RL_IC 无法超越 0.09;
    #   现纳入全部 Top-K (共 31 维), RL 潜力大幅提升
    actor_feats = topk + ["lgbm_sig"]
    obs_dim = len(actor_feats)

    # ── 3. 构造 CTDE episode (按日分组) ──
    print("[3/5] 构造 CTDE episodes + 训练...")
    episodes = []
    ep_dates = []
    # 标签中性化: 把 label 对风格/行业暴露做截面 OLS 残差化,
    # 对齐官方评测"风格剔除(取残差)", 消除 RL 偷吃市值/行业暴露的虚高 IC
    style_cols = [c for c in actor_feats if c in STYLE_HINTS]
    print(f"  风格中性化基准列: {style_cols if style_cols else '(无, 跳过)'}")

    for d, grp in tr_df.groupby("date"):
        grp = grp.dropna(subset=actor_feats + ["label"])
        if len(grp) < 10:
            continue
        obs = grp[actor_feats].values.astype(np.float32)
        # 中性化 label (对风格基准做 OLS 残差)
        if style_cols:
            S = grp[style_cols].values.astype(np.float32)
            S = (S - S.mean(0)) / (S.std(0) + 1e-6)
            A = np.column_stack([np.ones(len(S)), S])
            y = grp["label"].values.astype(np.float32)
            coef, *_ = np.linalg.lstsq(A, y, rcond=None)
            ret_neu = y - A @ coef
        else:
            ret_neu = grp["label"].values.astype(np.float32)
        gv = _global_vector(obs)
        episodes.append({
            "obs": torch.tensor(obs),
            "ret": torch.tensor(ret_neu),
            "global": torch.tensor(gv),
            "lgbm": torch.tensor(grp["lgbm_sig"].values.astype(np.float32)),
        })
        ep_dates.append(d)
    print(f"  Episodes (交易日): {len(episodes)}, obs_dim={obs_dim}")

    # ── 训练集内按时间划分验证期 (最后 20% 日期) 用于早停/权重, 防过拟合 ──
    split_idx = int(len(ep_dates) * 0.8)
    train_episodes = episodes[:split_idx]
    val_episodes = episodes[split_idx:]
    print(f"  训练期: {len(train_episodes)} 日 | 验证期: {len(val_episodes)} 日 "
          f"({ep_dates[0].date()}..{ep_dates[split_idx-1].date()} | "
          f"{ep_dates[split_idx].date()}..{ep_dates[-1].date()})")

    actor = DecentralizedActor(obs_dim)
    critic = CentralizedCritic(obs_dim * 7)
    actor, critic, best_val_ic = ctde_train(
        actor, critic, train_episodes, val_episodes=val_episodes,
        epochs=CTDE_EPOCHS, device=device,
        lr_decay_epoch=LR_DECAY_EPOCH, lr_decay=LR_DECAY,
        sig_tighten_epoch=SIG_TIGHTEN_EPOCH, patience=EARLY_STOP_PATIENCE,
    )

    # ── 4. 测试集预测 (分散执行, 仅 actor) ──
    print("[4/5] 预测测试集 (分散执行)...")
    fin_tbl = _resolve_table(datasources, "financial", FIN_TABLE,
                             "bigalpha_2026_financial") or FIN_TABLE
    bar_tbl = _resolve_table(datasources, "bar1m", BAR1M_TABLE,
                             "bigalpha_2026_stock_bar1m") or BAR1M_TABLE
    te_df, _ = build_features(fin_tbl, bar_tbl, start_date, end_date)
    te_df = _cs_winsorize(te_df, raw_feats)
    te_df = _cs_zscore(te_df, raw_feats)
    for c in raw_feats:
        te_df["r_" + c] = te_df.groupby("date")[c].rank(pct=True)
    # 测试集: slim LGBM 固定模型 predict (诚实样本外, 不泄漏)
    te_df = add_lgbm_signal(te_df, slim_model, topk)
    te_df = te_df.dropna(subset=actor_feats).reset_index(drop=True)

    actor.eval()
    with torch.no_grad():
        obs_t = torch.tensor(te_df[actor_feats].values.astype(np.float32))
        mu, _ = actor(obs_t)
        factor = mu.numpy().flatten()   # 直接用确定性 μ 作因子 (与训练奖励一致, 保持单调/不动秩)

    # ── 5. 截面标准化 + 对齐成分股 (v9 范式: 两通道各自吐合规 DataFrame) ──
    print("[5/5] 对齐成分股并输出...")

    # 符号校正 (确保因子越大→下期收益越高)
    tr_pred_all, tr_ret_all = _infer_train(actor, episodes)
    rl_sign = 1.0
    if len(tr_pred_all) > 1:
        s = np.sign(np.corrcoef(tr_pred_all, tr_ret_all)[0, 1])
        if not np.isnan(s) and s < 0:
            rl_sign = -1.0
    # LGBM 信号方向 (用训练集 OOF lgbm_sig 与 label 相关性)
    lgbm_sign = 1.0
    _m = ~np.isnan(tr_df["lgbm_sig"].values) & ~np.isnan(tr_df["label"].values)
    if _m.sum() > 1:
        s = np.sign(np.corrcoef(tr_df["lgbm_sig"].values[_m], tr_df["label"].values[_m])[0, 1])
        if not np.isnan(s) and s < 0:
            lgbm_sign = -1.0

    te_df["factor_rl"] = rl_sign * factor                        # RL 因子 (主)
    te_df["factor_lgbm"] = lgbm_sign * te_df["lgbm_sig"].values   # 纯 OOF LGBM 信号 (保底)

    # ── 本地 OOS 代理 IC 诊断 (无需上平台) ──
    # 测试集自带 label=T+1 收益, 直接算 RL 因子 / LGBM 因子的样本外 IC 及两者相关性.
    #   rl_oos↑ 且 corr(RL,LGBM)≈0 → RL 与 LGBM 正交(有独立 alpha, 加分); corr≈1 → RL 在复制 LGBM(冗余).
    _m = te_df["factor_rl"].notna() & te_df["label"].notna()
    if _m.sum() > 10:
        rl_oos = spearmanr(te_df.loc[_m, "factor_rl"], te_df.loc[_m, "label"])[0]
        lg_oos = spearmanr(te_df.loc[_m, "factor_lgbm"], te_df.loc[_m, "label"])[0]
        c_rl_lg = np.corrcoef(te_df.loc[_m, "factor_rl"], te_df.loc[_m, "factor_lgbm"])[0, 1]
        print(f"  [本地OOS代理] RL_IC={rl_oos:.4f}  LGBM_IC={lg_oos:.4f}  corr(RL,LGBM)={c_rl_lg:.4f}")
        print(f"    → corr≈0 表示 RL 学到真残差(加分); corr≈1 表示 RL 仍在复制 LGBM(无效)")

    # 查询成分股池 (仅查一次, 两通道复用)
    stk_pool = dai.query(
        f"SELECT date::DATE::DATETIME AS date, instrument::string AS instrument FROM {INST_TABLE}",
        filters={"date": [start_date, end_date]}, compression=True,
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"])
    stk_pool["instrument"] = stk_pool["instrument"].astype(str)

    # ★ v9 范式: 三通道各自吐出合规 [date,instrument,factor] DataFrame (已对齐成分股)
    rl_result = _build_factor_output(te_df, "factor_rl", stk_pool, tag="RL")
    lgbm_result = _build_factor_output(te_df, "factor_lgbm", stk_pool, tag="LGBM")

    # ── 复合因子: 在验证集上直接搜索 λ 使组合 IC(lgbm_z + λ·rl_z, ret) 最大 ──
    # （对齐"组合奖励"训练目标; 比按单通道 IC 加权更诚实, 直接最大化提交因子 OOS IC）
    _rl_pred, _lg_pred, _vr = [], [], []
    actor.eval()
    with torch.no_grad():
        for e in val_episodes:
            mu, _ = actor(e["obs"].to(device))
            _rl_pred.append(mu.cpu().numpy().flatten())
            _lg_pred.append(e["lgbm"].numpy().flatten())
            _vr.append(e["ret"].numpy().flatten())
    if _vr:
        _rl_pred = np.concatenate(_rl_pred); _lg_pred = np.concatenate(_lg_pred); _vr = np.concatenate(_vr)
        _rl_z = (_rl_pred - _rl_pred.mean()) / (_rl_pred.std() + 1e-6)
        _lg_z = (_lg_pred - _lg_pred.mean()) / (_lg_pred.std() + 1e-6)
        _best_lam, _best_cic = 0.0, -1.0
        for lam in np.linspace(0.0, 2.0, 21):
            _cic = spearmanr(_lg_z + lam * _rl_z, _vr)[0]
            if _cic > _best_cic:
                _best_cic = _cic; _best_lam = lam
        val_rl_ic = spearmanr(_rl_pred, _vr)[0]    # 仍报告单通道 IC (诊断用)
        val_lgbm_ic = spearmanr(_lg_pred, _vr)[0]
        w_rl = _best_lam / (1.0 + _best_lam)
        w_lgbm = 1.0 / (1.0 + _best_lam)
        print(f"  组合融合最优 λ={_best_lam:.2f} (val 组合 IC={_best_cic:.4f}) → "
              f"w_rl={w_rl:.3f} (val RL_IC={val_rl_ic:.4f}) "
              f"w_lgbm={w_lgbm:.3f} (val LGBM_IC={val_lgbm_ic:.4f})")
    else:
        val_rl_ic = val_lgbm_ic = 0.0
        w_rl = w_lgbm = 0.5
        print("  验证集为空, 退化为等权复合")
    if np.isnan(val_rl_ic): val_rl_ic = 0.0
    if np.isnan(val_lgbm_ic): val_lgbm_ic = 0.0
    te_df["factor_composite"] = w_rl * te_df["factor_rl"] + w_lgbm * te_df["factor_lgbm"]
    composite_result = _build_factor_output(te_df, "factor_composite", stk_pool, tag="COMPOSITE")

    # ── 因子选择 (默认 composite) ──
    result = composite_result
    if FACTOR_MODE == "rl":
        result = rl_result
    elif FACTOR_MODE == "lgbm":
        result = lgbm_result
    elif rl_result["factor"].std() < 1e-4 and lgbm_result["factor"].std() >= 1e-4:
        result = lgbm_result   # RL 坍缩时自动退回保底 OOF LGBM
        print("  ⚠️ RL 因子坍缩, 自动退回保底 OOF LGBM 信号因子")

    # ── 权重图: Top-K 特征 LGBM gain 重要性 + 复合融合权重 ──
    _plot_weights(full_model, rank_feats, topk, w_rl, w_lgbm, val_rl_ic, val_lgbm_ic)

    print("\n--- 提交因子预览 (前 10 行) ---")
    print(result[["date", "instrument", "factor"]].head(10).to_string(index=False))
    return result[["date", "instrument", "factor"]]


def _infer_train(actor, episodes):
    actor.eval()
    preds, rets = [], []
    with torch.no_grad():
        for e in episodes:
            mu, _ = actor(e["obs"])
            preds.append(mu.numpy().flatten())
            rets.append(e["ret"].numpy().flatten())
    return np.concatenate(preds), np.concatenate(rets)


if __name__ == "__main__":
    try:
        _sources = {"bar1m": BAR1M_TABLE, "financial": FIN_TABLE}
        _sd, _ed = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
        _res = main(_sources, _sd, _ed)
        print(f"\nDone. {len(_res)} rows.")
        print(_res.head(10))   # ← 直接打印因子值前 10 行
        # 平台提交: 调用官方评测接口让平台识别因子 (v9 同款)
        # ★ 关键: factor_pool 必须是 DataFrame (dai.query 结果), 传字符串会触发
        #   v4._normalize_date 的 "string indices must be integers" 报错!
        if M is not None and hasattr(M, "bigalpha_eval"):
            _factor_pool = dai.query(
                "SELECT * FROM bigalpha_2026_factorlib",
                filters={"date": [_sd, _ed]},
            ).df()
            M.bigalpha_eval._latest(
                factor_data=_res,
                factor_pool=_factor_pool,
                process_pools=False,
                show=True,
            )
    except Exception as e:
        print(f"Local test skipped: {e}")
        import traceback; traceback.print_exc()
        print("Paste this script into BigQuant Notebook to run on platform.")
